In [ ]:
import gc
import torch

from kernels.matmul_bf16 import matmul

In [ ]:
def get_vram(): 
    return torch.cuda.memory_allocated() / (1024**2)
def get_peak(): 
    return torch.cuda.max_memory_allocated() / (1024**2)

In [ ]:
def run_comparison(M=4096, K=4096, N=4096, dtype=torch.float16):
    print(f"тест на матрицах: A({M}x{K}) @ B({K}x{N}), тип {dtype}")
    
    # Создаем случайные матрицы
    A = torch.randn((M, K), device='cuda', dtype=dtype) / (K ** 0.5)
    B = torch.randn((K, N), device='cuda', dtype=dtype) / (K ** 0.5)
    
    gc.collect()
    torch.cuda.empty_cache()
    
    # Замер базовой памяти 
    base_vram = get_vram()

    # тестируем торч
    torch.cuda.reset_peak_memory_stats()
    
    C_torch = torch.matmul(A, B)
    torch.cuda.synchronize()
    
    torch_vram_diff = get_vram() - base_vram
    torch_peak_diff = get_peak() - base_vram
    
    del C_torch
    gc.collect()
    torch.cuda.empty_cache()

    # тестируем реализацию на тритоне
    torch.cuda.reset_peak_memory_stats()
    
    C_triton = matmul(A, B)
    torch.cuda.synchronize()
    
    triton_vram_diff = get_vram() - base_vram
    triton_peak_diff = get_peak() - base_vram

    # результат торча для сравнения ошибок
    C_torch = torch.matmul(A, B)

    # считаем ошибку
    C_torch_f32 = C_torch.to(torch.float32)
    C_triton_f32 = C_triton.to(torch.float32)
    
    mse = torch.nn.functional.mse_loss(C_torch_f32, C_triton_f32).item()

    print(f"{'Метрика':<25} | {'PyTorch (cuBLAS)':<15} | {'Triton':<15}")
    print(f"{'Выделено VRAM':<25} | {torch_vram_diff:>12.2f} MB | {triton_vram_diff:>12.2f} MB")
    print(f"{'Пиковое VRAM':<25} | {torch_peak_diff:>12.2f} MB | {triton_peak_diff:>12.2f} MB")
    print(f"MSE Ошибка:               {mse:.8e}")

In [ ]:
run_comparison(M=4096, K=4096, N=4096, dtype=torch.float16)

In [ ]:
run_comparison(M=1, K=4096, N=4096, dtype=torch.float16)